In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Training will be very slow.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.64 GB


In [3]:
import os
import sys
import shutil
from pathlib import Path

# Define paths
COLAB_ROOT = "/content"
PROJECT_NAME = "Colab_text_modelling_dp"
DRIVE_PATH = "/content/drive/MyDrive"

PROJECT_PATH = f"{COLAB_ROOT}/{PROJECT_NAME}"
DRIVE_PROJECT_PATH = f"{DRIVE_PATH}/{PROJECT_NAME}"

# Create directories
os.makedirs(PROJECT_PATH, exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT_PATH}/models", exist_ok=True)

print(f"Project path (Colab): {PROJECT_PATH}")
print(f"Project path (Drive backup): {DRIVE_PROJECT_PATH}")

# Change to project directory
os.chdir(PROJECT_PATH)

Project path (Colab): /content/Colab_text_modelling_dp
Project path (Drive backup): /content/drive/MyDrive/Colab_text_modelling_dp


In [4]:
print("📋 Copying entire project from Drive to Colab...")

# Check if project exists in Drive
import os
from pathlib import Path

drive_project = Path("/content/drive/MyDrive/Colab_text_modelling_dp")
colab_project = Path("/content/Colab_text_modelling_dp")

if drive_project.exists():
    print(f"✅ Found project in Drive: {drive_project}")

    # List contents
    print("\n📁 Drive project contents:")
    for item in drive_project.iterdir():
        print(f"  • {item.name}")

    # Copy everything
    print("\n🚀 Copying to Colab...")
    !cp -r "{drive_project}"/* "{colab_project}"/

    print(f"✅ Copied! Colab project now has:")
    for item in colab_project.iterdir():
        print(f"  • {item.name}")

else:
    print("❌ Project not found in Drive!")
    print(f"Expected at: {drive_project}")
    print("Please make sure you uploaded to:")
    print("Google Drive → MyDrive → lecture_summarization")

📋 Copying entire project from Drive to Colab...
✅ Found project in Drive: /content/drive/MyDrive/Colab_text_modelling_dp

📁 Drive project contents:
  • CLAUDE.md
  • config
  • data
  • implementation_plan.md.resolved
  • logs
  • models
  • requirements.txt
  • results
  • scripts
  • walkthrough.md.resolved
  • wandb

🚀 Copying to Colab...
✅ Copied! Colab project now has:
  • CLAUDE.md
  • scripts
  • implementation_plan.md.resolved
  • wandb
  • requirements.txt
  • walkthrough.md.resolved
  • results
  • logs
  • data
  • models
  • config


In [5]:
!pip install -r "/content/Colab_text_modelling_dp/requirements.txt"

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-85hwc4a_
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-85hwc4a_
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 82.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 25.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 21.4 MB/s eta 0:00:00
     ━

In [ ]:
file_path = "data/processed/bart_dataset/tokenizer"
full_path = os.path.join(PROJECT_PATH, file_path)
if os.path.exists(full_path):
    print(f"✓ {file_path}")
else:
    print(f"✗ {file_path} - MISSING!")

✓ data/processed/bart_dataset/tokenizer


In [ ]:
print("=" * 60)
print("Running prepare dataset for BART")
print("=" * 60)

!cd {PROJECT_PATH} && python scripts/text_processing/05_prepare_bart_data.py

Running prepare dataset for BART
2026-03-12 14:48:42 - bart_data_prep - INFO - BARTDataPreparator initialized (test_mode=False)
2026-03-12 14:48:42 - bart_data_prep - INFO - ============================================================
2026-03-12 14:48:42 - bart_data_prep - INFO - Starting BART Data Preparation
2026-03-12 14:48:42 - bart_data_prep - INFO - ============================================================
2026-03-12 14:48:42 - bart_data_prep - INFO - Loading 239 annotation files ...
Loading annotations: 100% 239/239 [00:02<00:00, 94.72it/s]
2026-03-12 14:48:45 - bart_data_prep - INFO - Successfully loaded 239 annotations
2026-03-12 14:48:45 - bart_data_prep - WARNING - segmentation_results.json not found – using fallback chunking
Building segments: 100% 239/239 [00:00<00:00, 1393.77it/s]
2026-03-12 14:48:45 - bart_data_prep - INFO - Segments created: 1421
2026-03-12 14:48:45 - bart_data_prep - INFO - Avg words/segment: 372.9
2026-03-12 14:48:45 - bart_data_prep - INFO - Skipp

In [ ]:
print("=" * 60)
print("Generating pseudo summaries for segments")
print("=" * 60)

!cd {PROJECT_PATH} && python scripts/text_processing/06_generate_pseudo_summaries.py

Generating pseudo summaries for segments
2026-03-12 14:49:58 - pseudo_summary_gen - INFO - ============================================================
2026-03-12 14:49:58 - pseudo_summary_gen - INFO - Starting Pseudo-Summary Generation
2026-03-12 14:49:58 - pseudo_summary_gen - INFO - ============================================================
2026-03-12 14:50:03 - pseudo_summary_gen - INFO - Loading FLAN-T5: google/flan-t5-large …
tokenizer_config.json: 2.54kB [00:00, 12.8MB/s]
spiece.model: 100% 792k/792k [00:00<00:00, 1.87MB/s]
tokenizer.json: 2.42MB [00:00, 52.9MB/s]
special_tokens_map.json: 2.20kB [00:00, 12.5MB/s]
config.json: 100% 662/662 [00:00<00:00, 5.28MB/s]
E0000 00:00:1773327010.964652    3492 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773327011.038667    3492 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when

In [ ]:
print("=" * 60)
print("Creating dataset for bart")
print("=" * 60)

!cd {PROJECT_PATH} && python scripts/text_processing/create_bart_dataset.py

Creating dataset for bart
2026-03-12 17:14:40 - bart_dataset - INFO - ============================================================
2026-03-12 17:14:40 - bart_dataset - INFO - Building BART Dataset
2026-03-12 17:14:40 - bart_dataset - INFO - ============================================================
2026-03-12 17:14:42 - bart_dataset - INFO - Loading tokenizer: facebook/bart-large-cnn
config.json: 1.58kB [00:00, 10.0MB/s]
vocab.json: 899kB [00:00, 24.4MB/s]
merges.txt: 456kB [00:00, 95.1MB/s]
tokenizer.json: 1.36MB [00:00, 102MB/s]
2026-03-12 17:14:43 - bart_dataset - INFO - Added 7 special tokens to tokenizer
2026-03-12 17:14:43 - bart_dataset - INFO - Tokenizer saved → data/processed/bart_dataset/tokenizer
2026-03-12 17:14:43 - bart_dataset - INFO - Creating entity masks for entity-aware loss …
Entity masks:   0% 0/1421 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lam

In [ ]:
print("=" * 60)
print("Training BART Model")
print("=" * 60)

!cd {PROJECT_PATH} && python scripts/text_processing/08_train_bart_summarizer.py --test

Training BART Model
2026-03-11 07:11:04 - bart_trainer - INFO - Device: cuda
2026-03-11 07:11:04 - bart_trainer - INFO - GPU: Tesla T4
2026-03-11 07:11:04 - bart_trainer - INFO - VRAM: 15.6 GB
2026-03-11 07:11:04 - bart_trainer - INFO - ============================================================
2026-03-11 07:11:04 - bart_trainer - INFO - Starting BART Fine-Tuning
2026-03-11 07:11:04 - bart_trainer - INFO - ============================================================
2026-03-11 07:11:09.433273: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773213069.650004    9914 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773213069.710457    9914 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS w

In [6]:
print("=" * 60)
print("Training BART Model")
print("=" * 60)

!cd {PROJECT_PATH} && python scripts/text_processing/08_train_bart_summarizer.py

Training BART Model
2026-03-13 05:37:06 - bart_trainer - INFO - Device: cuda
2026-03-13 05:37:06 - bart_trainer - INFO - GPU: Tesla T4
2026-03-13 05:37:06 - bart_trainer - INFO - VRAM: 15.6 GB
2026-03-13 05:37:06 - bart_trainer - INFO - ============================================================
2026-03-13 05:37:06 - bart_trainer - INFO - Starting BART Fine-Tuning
2026-03-13 05:37:06 - bart_trainer - INFO - ============================================================
2026-03-13 05:37:16.936647: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773380237.157119    4454 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773380237.215621    4454 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS w

In [ ]:
# ONE LINE - Overwrites Drive folder with Colab changes
!rm -rf /content/drive/MyDrive/Colab_text_modelling_dp && cp -r /content/Colab_text_modelling_dp /content/drive/MyDrive/
print("✅ Drive folder updated with Colab changes!")

✅ Drive folder updated with Colab changes!
